Stock Safety - Current + Incoming Stock vs. Forecasted Demand
Compares available stock (on hand + open purchase orders) against item-level monthly demand, allocated per location by historical sales share. Flags Safe / Not Safe.

In [0]:
%run ../../_local_config

In [0]:
import sys
sys.path.append("/Workspace/Users/venura-it@brownsgroup.com/Exide sales/Exide-Sales-Forecast")

from src.io.storage import get_blob_service, read_silver, read_bronze, save_json
from src.analysis.stock_safety import (
    compute_current_stock, compute_incoming_stock,
    allocate_item_forecast_to_locations, compute_stock_safety
)
import pandas as pd
import json
import io

blob_service = get_blob_service(storage_account_name, storage_account_key)

FORECAST_BASE = "live/battery/forecasts"
ANALYSIS_BASE = "live/battery/analysis"

Load 

In [0]:
analysis_silver = read_silver(blob_service, f"{ANALYSIS_BASE}/battery_analysis_clean_live.json")
analysis_silver["posting_date"] = pd.to_datetime(analysis_silver["posting_date"])

purchase_orders_bronze = read_bronze(blob_service, "live/battery/purchase_orders_open.json")

blob_client = blob_service.get_blob_client(container="gold", blob=f"{FORECAST_BASE}/active/item_monthly_forecast_active.json")
item_monthly_active = pd.DataFrame(json.loads(blob_client.download_blob().readall()))

print(f"Analysis silver: {analysis_silver.shape}")
print(f"Purchase orders: {purchase_orders_bronze.shape}")
print(f"Item monthly forecast (active): {item_monthly_active.shape}")

Compute current + incoming stock

In [0]:
battery_item_list = analysis_silver[analysis_silver["itemCategoryCode"] == "BATTERY"]["item_no"].unique().tolist()

current_stock = compute_current_stock(analysis_silver)
incoming_stock = compute_incoming_stock(purchase_orders_bronze, battery_item_list)

print(f"Current stock rows: {len(current_stock)}")
print(f"Incoming stock rows: {len(incoming_stock)}")
print(incoming_stock.sort_values("incoming_stock", ascending=False).head(10))

Allocate forecast to locations, using the NEXT month's forecast

In [0]:
next_month_forecast = item_monthly_active.sort_values("generated_date").drop_duplicates(subset="item_no", keep="last")
next_month_forecast = next_month_forecast.rename(columns={"predicted_units": "predicted_units"})

allocated_forecast = allocate_item_forecast_to_locations(next_month_forecast, analysis_silver)
print(allocated_forecast.head(10))

Compute stock safety

In [0]:
stock_safety = compute_stock_safety(current_stock, incoming_stock, allocated_forecast)

print(stock_safety["status"].value_counts())
print("\nNot Safe items (most urgent first):")
print(stock_safety[stock_safety["status"] == "Not Safe"][
    ["item_no", "location_code", "current_stock", "incoming_stock", "available_stock", "location_forecast_units", "safety_ratio"]
].head(30))

Save

In [0]:
save_json(blob_service, stock_safety, f"{ANALYSIS_BASE}/stock_safety.json")

buffer = io.BytesIO()
stock_safety.to_excel(buffer, index=False, engine="openpyxl")
buffer.seek(0)
blob_client = blob_service.get_blob_client(container="gold", blob=f"{ANALYSIS_BASE}/stock_safety.xlsx")
blob_client.upload_blob(buffer, overwrite=True)
print("Saved stock_safety.json and .xlsx")